# PCA + Decision Tree: Predicting Steel Microstructure

**Goal:** Use a small table of material-processing characteristics to predict a simplified microstructure label: **Ferrite**, **Pearlite**, or **Martensite**. Then reduce the number of features using **PCA** before training a **decision tree**.

> **Teaching example only:** The included CSV contains synthetic, intentionally simplified data. Real steel microstructures depend on composition, heat-treatment history, phase kinetics, and often contain mixtures; this is **not** a validated metallurgical predictor.
**Files:** Keep `material_microstructure_data.csv` in the same folder as this notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report

# Keep material_microstructure_data.csv in the same folder as this notebook.
df = pd.read_csv("material_microstructure_data.csv")
columns = ["carbon_wt_pct", "cooling_rate_C_per_s", "austenitize_C",
           "manganese_wt_pct", "prior_grain_um"]

# Check the required columns are present.
required = columns + ["microstructure"]
missing = [name for name in required if name not in df.columns]
if missing:
    raise ValueError(f"Missing CSV columns: {missing}")

display(df.groupby("microstructure").head(3).round(2))
print("Dataset shape:", df.shape)


## 1. Load the CSV and split the data
The previous cell loads `material_microstructure_data.csv`. We reserve **25% of samples for testing**. To avoid data leakage, we examine correlations on the **training features only** and fit the PCA preprocessing only on training data.

In [ ]:
X = df[columns]
y = df["microstructure"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Training rows: {len(X_train)}; test rows: {len(X_test)}")

## 2. Explore correlations before PCA
The **correlation matrix** shows how strongly pairs of material characteristics vary together. Values run from **−1** (strong negative relationship) to **+1** (strong positive relationship); **0** means little *linear* relationship. Each feature has a correlation of +1 with itself.

Strong correlations can indicate overlapping information between features—a reason PCA might be helpful. Correlation does **not** prove causation, and PCA can still be useful even when correlations are modest.

In [ ]:
# Pearson correlation matrix of the original training features (before scaling/PCA)
corr = X_train.corr()
display(corr.round(2))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(columns)), labels=columns, rotation=45, ha="right")
ax.set_yticks(range(len(columns)), labels=columns)
for row in range(len(columns)):
    for col in range(len(columns)):
        val = corr.iloc[row, col]
        ax.text(col, row, f"{val:.2f}", ha="center", va="center",
                color="white" if abs(val) > 0.65 else "black", fontsize=9)
fig.colorbar(im, ax=ax, label="Pearson correlation (−1 to +1)")
ax.set_title("Material feature correlations (training data)")
plt.tight_layout()
plt.show()

## 3. Reduce 5 features to 2 PCs and train a decision tree
We **standardize** the features, use **PCA** to turn the 5 original features into 2 principal components, and train the tree on those components. The pipeline fits preprocessing using training samples only.

In [ ]:
pca_tree = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("tree", DecisionTreeClassifier(max_depth=3, random_state=42)),
])
pca_tree.fit(X_train, y_train)
y_pred = pca_tree.predict(X_test)
print(f"PCA + tree test accuracy: {accuracy_score(y_test, y_pred):.1%}")
print(classification_report(y_test, y_pred, zero_division=0))

## 4. What did PCA do?
The original table has **5 numeric features**. PCA combines them into **2 new features**, called **PC1** and **PC2**. They are directions that preserve as much variation as possible—not necessarily the information most useful for classification.

In [ ]:
scaler = pca_tree.named_steps["scale"]
pca = pca_tree.named_steps["pca"]
X_train_pc = pca.transform(scaler.transform(X_train))

print("Variance explained by PC1 and PC2:", np.round(pca.explained_variance_ratio_, 3))
print("Total variance retained:", f"{pca.explained_variance_ratio_.sum():.1%}")

fig, ax = plt.subplots(figsize=(6, 4))
for label in sorted(y_train.unique()):
    mask = y_train.to_numpy() == label
    ax.scatter(X_train_pc[mask, 0], X_train_pc[mask, 1], label=label, alpha=0.8)
ax.set(xlabel="PC1", ylabel="PC2", title="Training materials projected onto 2 PCs")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Inspect the decision tree and test predictions
The tree asks simple **yes/no threshold questions about PC1 and PC2**. It no longer sees the 5 original columns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_tree(
    pca_tree.named_steps["tree"],
    feature_names=["PC1", "PC2"],
    class_names=pca_tree.named_steps["tree"].classes_,
    filled=True, rounded=True, fontsize=8, ax=axes[0]
)
axes[0].set_title("Decision tree after PCA")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[1], xticks_rotation=25, colorbar=False)
axes[1].set_title("Unseen test rows")
plt.tight_layout()
plt.show()

## 6. Compare with a tree that skips PCA
PCA is useful for reducing dimensions, but it **does not guarantee better classification**. Compare performance using the same train/test split.

In [ ]:
original_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
original_tree.fit(X_train, y_train)
print(f"Tree using all 5 features: {original_tree.score(X_test, y_test):.1%}")
print(f"Tree using 2 PCs:        {pca_tree.score(X_test, y_test):.1%}")

# Try it yourself: change n_components=2 to 3, or vary max_depth.
# Remember: accuracy here reflects only this synthetic teaching dataset.

### Key takeaway
**Material data (5 columns) → correlation check → standardize → PCA (2 columns) → decision tree → microstructure label.**

In a real project, use measured data, physically meaningful features, cross-validation, and a truly independent test set. Do not interpret these example accuracy scores as real-world predictive performance.